# SEO Inspector — Common Crawl referring domains (watchlist import)

Builds the **full list of referring domains** for every domain on your SEO Inspector watchlist
(clients and competitors) from the Common Crawl **domain-level web graph**, and uploads it to the app.
The app then shows referring-domain counts, new/lost domains between releases and the competitor link gap.

How it works (nothing large is stored — files are streamed):
1. Read your watchlist from the app.
2. Stream the graph's *vertices* file (~1 GB gzip) to find the node id of each watchlist domain.
3. Stream the *edges* file (~7–8 GB gzip) through `awk`, keeping only edges that point at those ids.
4. Stream the vertices file again to turn the linking ids back into domain names.
5. Upload one list per watchlist domain to the app.

**Runtime:** roughly 20–60 minutes on a standard Colab runtime (mostly download speed).
Run it after each new Common Crawl release (about monthly; the app keeps the current and previous one).

**You need:** your app URL and the `IMPORT_API_TOKEN` secret configured on the app.

In [ ]:
# --- Settings -------------------------------------------------------------
APP_URL = "https://your-app.replit.app"   # your SEO Inspector URL, no trailing slash
RELEASE = ""                              # leave empty to use the latest Common Crawl web-graph release

import getpass
IMPORT_TOKEN = getpass.getpass("IMPORT_API_TOKEN: ")

In [ ]:
import datetime, json, os, subprocess, urllib.request

BASE = "https://data.commoncrawl.org/projects/hyperlinkgraph"
MONTHS = ["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"]

def head_ok(url):
    req = urllib.request.Request(url, method="HEAD")
    try:
        return urllib.request.urlopen(req, timeout=20).status == 200
    except Exception:
        return False

def latest_release():
    try:
        info = json.load(urllib.request.urlopen("https://index.commoncrawl.org/graphinfo.json", timeout=20))
        return info[0]["id"]
    except Exception:
        pass
    d = datetime.date.today(); y, m = d.year, d.month
    for _ in range(18):                      # probe the naming scheme, newest first
        idx = [(m - 3 + k) % 12 for k in range(3)]
        start = y if m >= 3 else y - 1
        yy = f"{start}-{str(start + 1)[2:]}" if idx[0] > idx[2] else f"{y}"
        rel = f"cc-main-{yy}-{'-'.join(MONTHS[i] for i in idx)}"
        if head_ok(f"{BASE}/{rel}/domain/{rel}-domain-ranks.txt.gz"):
            return rel
        m -= 1
        if m == 0:
            m, y = 12, y - 1
    raise RuntimeError("No Common Crawl web-graph release found")

RELEASE = RELEASE or latest_release()
VERTICES = f"{BASE}/{RELEASE}/domain/{RELEASE}-domain-vertices.txt.gz"
EDGES = f"{BASE}/{RELEASE}/domain/{RELEASE}-domain-edges.txt.gz"
print("Release:", RELEASE)

In [ ]:
# --- 1. Watchlist from the app --------------------------------------------
req = urllib.request.Request(f"{APP_URL}/api/import/watchlist", headers={"Authorization": f"Bearer {IMPORT_TOKEN}"})
watch = json.load(urllib.request.urlopen(req, timeout=30))["domains"]
print(len(watch), "watchlist domains:", watch)

def rev(domain):                       # example.co.uk -> uk.co.example (graph format)
    return ".".join(reversed(domain.lower().removeprefix("www.").split(".")))

with open("targets_rev.txt", "w") as f:
    f.write("\n".join(rev(d) for d in watch) + "\n")

In [ ]:
# --- 2. Vertex ids of the watchlist domains (streams ~1 GB) ---------------
cmd = (f"curl -s '{VERTICES}' | gzip -dc | "
       "awk -F'\\t' 'NR==FNR{want[$1]=1; next} ($2 in want){print $1\"\\t\"$2}' targets_rev.txt - > target_ids.tsv")
subprocess.run(cmd, shell=True, check=True)
target_ids = dict(line.rstrip("\n").split("\t") for line in open("target_ids.tsv"))  # id -> rev domain
missing = set(rev(d) for d in watch) - set(target_ids.values())
print(f"found {len(target_ids)} of {len(watch)} domains in the graph; not in graph: {sorted(missing) or 'none'}")
with open("target_ids.txt", "w") as f:
    f.write("\n".join(target_ids) + "\n")

In [ ]:
# --- 3. Edges pointing at the targets (streams ~7-8 GB; the slow step) ----
cmd = (f"curl -s '{EDGES}' | gzip -dc | "
       "awk -F'\\t' 'NR==FNR{t[$1]=1; next} ($2 in t){print $1\"\\t\"$2}' target_ids.txt - > inlinks_ids.tsv")
subprocess.run(cmd, shell=True, check=True)
print(sum(1 for _ in open("inlinks_ids.tsv")), "linking edges found")

In [ ]:
# --- 4. Names of the linking domains (streams ~1 GB again) ----------------
subprocess.run("cut -f1 inlinks_ids.tsv | sort -u > source_ids.txt", shell=True, check=True)
cmd = (f"curl -s '{VERTICES}' | gzip -dc | "
       "awk -F'\\t' 'NR==FNR{s[$1]=1; next} ($1 in s){print $1\"\\t\"$2}' source_ids.txt - > source_names.tsv")
subprocess.run(cmd, shell=True, check=True)
names = dict(line.rstrip("\n").split("\t") for line in open("source_names.tsv"))  # id -> rev domain

def unrev(r):
    return ".".join(reversed(r.split(".")))

per_target = {}
for line in open("inlinks_ids.tsv"):
    src, dst = line.rstrip("\n").split("\t")
    if src in names:
        per_target.setdefault(unrev(target_ids[dst]), set()).add(unrev(names[src]))
for t, s in sorted(per_target.items()):
    print(f"{t}: {len(s)} referring domains")

In [ ]:
# --- 5. Upload to the app ---------------------------------------------------
for target, sources in per_target.items():
    body = json.dumps({"release": RELEASE, "target_domain": target, "source_domains": sorted(sources)}).encode()
    req = urllib.request.Request(f"{APP_URL}/api/import/inlinks", data=body, method="POST",
                                 headers={"Authorization": f"Bearer {IMPORT_TOKEN}", "Content-Type": "application/json"})
    print(json.load(urllib.request.urlopen(req, timeout=300)))
print("Done. Re-run an audit to see the full referring-domain data.")